# Week 8 - Single Agent Systems & Agent Pipelines

## Small Agent Pipeline Project

### Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### Required Routing
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

### Implementation Requirements
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### Bonus Improvements Included
- Improved routing
- Basic logging
- Safer calculator implementation
- Clear test cases
- Error-handling tests
- Execution summary

> This notebook is based on the instructor-provided Week 8 assignment notebook and completes the unfinished agent logic rather than replacing the project with a different architecture.

## 1. Understanding the Agent Pipeline

A single-agent pipeline can be viewed as a small stateful workflow:

**User Query → Intent Detection → Conditional Routing → Tool / General Response → Structured Output**

The agent decides which action is appropriate for the incoming query. This demonstrates conditional routing and tool integration in a simple single-agent system.

In [1]:
import ast
import operator as op
import json
import logging
import re

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

print("Required libraries imported successfully.")

Required libraries imported successfully.


## 2. Tool 1 - Calculator

The original assignment provides a Calculator Tool. Here it is implemented with basic error handling.

For the project implementation, arithmetic expressions are evaluated through a restricted AST-based calculator rather than unrestricted `eval()`. This keeps the calculator limited to arithmetic operations.

In [2]:
# 🛠️ TOOL 1: Calculator

_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
    ast.USub: op.neg,
    ast.UAdd: op.pos,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))

    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        left = _safe_eval(node.left)
        right = _safe_eval(node.right)
        return _ALLOWED_OPERATORS[type(node.op)](left, right)

    raise ValueError("Only basic arithmetic expressions are supported.")

def calculator(expression: str) -> str:
    """Evaluate a basic mathematical expression safely."""
    try:
        expression = expression.strip()
        if not expression:
            raise ValueError("Empty expression.")

        # Keep the calculator focused on arithmetic expressions.
        if not re.fullmatch(r"[0-9.()\s+\-*/%]+", expression):
            raise ValueError("Unsupported characters in expression.")

        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)

        return str(result)

    except ZeroDivisionError:
        return "Error in calculation: division by zero"

    except Exception as e:
        logging.warning("Calculator error: %s", e)
        return "Error in calculation"

## 3. Tool 2 - Keyword Extractor

The instructor-provided Keyword Extractor identifies words longer than four characters and returns up to five keywords.

In [3]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract up to five simple keywords from text."""
    try:
        words = text.split()
        keywords = list(set(
            [w.lower() for w in words if len(w) > 4]
        ))
        return keywords[:5]

    except Exception as e:
        logging.warning("Keyword extraction error: %s", e)
        return []

## 4. Agent Routing Logic

The required conditional routing is:

- If the query contains **"calculate"** → Calculator Tool
- If the query contains **"keywords"** → Keyword Extractor Tool
- Otherwise → General Response

The agent returns a structured object with:

```text
{
    "type": "calculation / keywords / general / error",
    "result": ...
}
```

For clarity in the notebook, the final response is printed as JSON.

In [4]:
# 🤖 AGENT FUNCTION

def agent(query: str):
    """
    Single-agent smart assistant.

    Routes the query to the appropriate tool based on simple
    intent detection and returns structured output.
    """

    try:
        if not isinstance(query, str):
            return {
                "type": "error",
                "result": "Query must be a string."
            }

        query = query.strip()

        if not query:
            return {
                "type": "error",
                "result": "Query cannot be empty."
            }

        query_lower = query.lower()

        logging.info("Received query: %s", query)

        # Conditional route 1: Calculator
        if "calculate" in query_lower:
            expression = query_lower.split("calculate", 1)[1].strip()

            if not expression:
                return {
                    "type": "error",
                    "result": "No mathematical expression was provided."
                }

            result = calculator(expression)

            if result.startswith("Error"):
                return {
                    "type": "error",
                    "result": result
                }

            return {
                "type": "calculation",
                "result": result
            }

        # Conditional route 2: Keyword extractor
        elif "keywords" in query_lower:
            text = re.split(r"keywords", query, maxsplit=1, flags=re.IGNORECASE)[1].strip()

            if not text:
                return {
                    "type": "error",
                    "result": "No text was provided for keyword extraction."
                }

            result = extract_keywords(text)

            return {
                "type": "keywords",
                "result": result
            }

        # Conditional route 3: General response
        else:
            return {
                "type": "general",
                "result": (
                    "This is a general query. The single-agent pipeline "
                    "did not require a specialized tool."
                )
            }

    except Exception as e:
        logging.exception("Unexpected agent error")
        return {
            "type": "error",
            "result": str(e)
        }

## 5. Helper Function for Structured JSON Output

The agent internally returns a Python dictionary. This helper converts the result to valid JSON so the structured output is easy to inspect and reuse.

In [5]:
def run_agent(query: str):
    result = agent(query)
    print(json.dumps(result, indent=2))
    return result

## 6. Test Case 1 - Mathematical Query

This test checks whether a calculation query is routed to the Calculator Tool.

In [6]:
result_1 = run_agent("Calculate 20 + 5")

{
  "type": "calculation",
  "result": "25"
}


## 7. Test Case 2 - Keyword Extraction

This test checks whether a keyword query is routed to the Keyword Extractor Tool.

In [7]:
result_2 = run_agent(
    "Extract keywords from Artificial Intelligence is transforming industries"
)

{
  "type": "keywords",
  "result": [
    "industries",
    "artificial",
    "intelligence",
    "transforming"
  ]
}


## 8. Test Case 3 - General Query

Queries that do not match the two specialized routes are handled by the General Response module.

In [8]:
result_3 = run_agent("What is machine learning?")

{
  "type": "general",
  "result": "This is a general query. The single-agent pipeline did not require a specialized tool."
}


## 9. Error Handling Tests

The agent includes basic error handling for:
- Empty queries
- Missing calculator expressions
- Missing keyword-extraction text
- Invalid calculator expressions
- Division by zero
- Non-string input

In [9]:
error_tests = [
    "",
    "Calculate",
    "Keywords",
    "Calculate 10 / 0",
    "Calculate 10 + abc",
    None
]

for query in error_tests:
    print("=" * 60)
    print("Input:", repr(query))
    print("Output:")
    print(json.dumps(agent(query), indent=2))

Input: ''
Output:
{
  "type": "error",
  "result": "Query cannot be empty."
}
Input: 'Calculate'
Output:
{
  "type": "error",
  "result": "No mathematical expression was provided."
}
Input: 'Keywords'
Output:
{
  "type": "error",
  "result": "No text was provided for keyword extraction."
}
Input: 'Calculate 10 / 0'
Output:
{
  "type": "error",
  "result": "Error in calculation: division by zero"
}
Input: 'Calculate 10 + abc'
Output:
{
  "type": "error",
  "result": "Error in calculation"
}
Input: None
Output:
{
  "type": "error",
  "result": "Query must be a string."
}


## 10. Multiple Query Evaluation

The following test set evaluates all three main routing paths together.

In [10]:
queries = [
    "Calculate 20 + 5",
    "Calculate (10 * 3) - 4",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Extract keywords from Data Science and Machine Learning",
    "What is machine learning?",
    "Explain artificial intelligence"
]

evaluation_results = []

for q in queries:
    response = agent(q)

    evaluation_results.append({
        "query": q,
        "type": response["type"],
        "result": response["result"]
    })

    print("Query:", q)
    print("Response:", json.dumps(response, indent=2))
    print("-" * 60)

Query: Calculate 20 + 5
Response: {
  "type": "calculation",
  "result": "25"
}
------------------------------------------------------------
Query: Calculate (10 * 3) - 4
Response: {
  "type": "calculation",
  "result": "26"
}
------------------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {
  "type": "keywords",
  "result": [
    "industries",
    "artificial",
    "intelligence",
    "transforming"
  ]
}
------------------------------------------------------------
Query: Extract keywords from Data Science and Machine Learning
Response: {
  "type": "keywords",
  "result": [
    "machine",
    "science",
    "learning"
  ]
}
------------------------------------------------------------
Query: What is machine learning?
Response: {
  "type": "general",
  "result": "This is a general query. The single-agent pipeline did not require a specialized tool."
}
--------------------------------------------------

## 11. Routing Summary

The evaluation below counts how many queries were routed to each response type.

In [11]:
from collections import Counter

route_counts = Counter(
    item["type"] for item in evaluation_results
)

print("Routing Summary")
print("=" * 40)

for route, count in route_counts.items():
    print(f"{route}: {count}")

Routing Summary
calculation: 2
keywords: 2
general: 2


## 12. Agent Pipeline Representation

The completed pipeline can be represented as:

**User Query**
↓
**Agent**
↓
**Intent Detection**
↓
┌───────────────────┬──────────────────────┬───────────────────┐
↓                   ↓                      ↓
**Calculate**       **Keywords**           **General**
↓                   ↓                      ↓
Calculator Tool     Keyword Tool           Direct Response
└───────────────────┴──────────────────────┘
↓
**Structured JSON Output**

This demonstrates conditional routing and tool integration within a single-agent system.

## 13. Sequential vs Parallel Tool Calls

This project uses **sequential routing** because the agent first needs to understand the query and then select the appropriate tool.

Parallel calls would be useful when multiple independent tools need to process the same query at the same time. In this small project, only one specialized tool is required for each routed query.

## 14. Error Handling and Reliability

Basic error handling is implemented with `try-except` blocks and explicit validation.

The pipeline handles invalid or incomplete inputs without stopping the complete notebook. A separate error route is returned when an operation cannot be completed.

The retry-loop concept from the Week 8 quiz is relevant to larger systems where temporary failures, such as external API failures, may require another attempt.

## 15. Observations

- The agent successfully separates queries into calculation, keyword, and general routes.
- The Calculator Tool handles basic arithmetic expressions.
- The Keyword Extractor returns a small set of keywords from text.
- General questions are handled without invoking a specialized tool.
- Structured JSON output makes the result easier for another component to consume.
- Basic validation and exception handling improve robustness.

## 16. Limitations

This is intentionally a small rule-based single-agent pipeline.

- Intent detection is based mainly on keywords.
- The General Response module is not a full language model.
- The keyword extractor is a simple word-based approach.
- The calculator supports basic arithmetic rather than a complete mathematical language.
- The system does not maintain long-term conversational state.

## 17. Possible Improvements

The assignment's bonus ideas can be extended by:

1. Improving intent classification.
2. Adding more specialized tools.
3. Adding richer logging and monitoring.
4. Maintaining conversation state.
5. Adding retry mechanisms for temporary tool failures.
6. Supporting parallel tool calls when multiple independent operations are needed.
7. Adding a language model for more capable general responses.

## 18. Conclusion

A complete Single-Agent Smart Assistant was implemented using the instructor-provided project structure.

The agent uses conditional routing to decide whether a query should go to the Calculator Tool, Keyword Extractor Tool, or General Response module. It returns structured JSON-style results and includes basic error handling.

The project demonstrates the core ideas of single-agent workflows, tool integration, conditional routing, structured outputs, and reliability.

## 19. Viva Questions and Answers

### Q1. What is a single-agent system?
A single-agent system is a system where one agent manages the task, decides which action is required, and uses available tools when necessary.

### Q2. What is conditional routing?
Conditional routing directs a query to different tools or actions based on its detected intent.

### Q3. What are nodes and edges in an agent workflow?
Nodes represent tasks or actions, while edges represent the connections that determine how information moves between tasks.

### Q4. Why are tools used by agents?
Tools allow an agent to perform specialized operations such as calculations, searches, or data processing.

### Q5. Why is structured JSON output useful?
It provides a consistent format that other software components can easily read and process.

### Q6. What is error handling in an agent?
Error handling allows the agent to respond appropriately when a tool fails or when invalid input is received.

### Q7. What is the difference between sequential and parallel tool calls?
Sequential calls run one after another, while parallel calls run independent operations at the same time.

### Q8. What is conditional routing in this project?
Queries containing "calculate" are routed to the Calculator Tool, queries containing "keywords" are routed to the Keyword Tool, and other queries use the General Response route.

### Q9. How can this project be improved?
It can be improved with better intent detection, more tools, conversation state, retry mechanisms, logging, and a language model.

### Q10. What is the main limitation of this project?
The routing is rule-based and therefore depends on simple keywords rather than a more advanced intent-classification system.